In [ ]:
# CELL 1 — ONLY USER CONTROL
RUN_SEED = 1337  # CHANGE ONLY THIS: 42 / 1337 / 2026 / 31415 / 271828

PUBLICATION_SEEDS = [42, 1337, 2026, 31415, 271828]
if RUN_SEED not in PUBLICATION_SEEDS:
    raise ValueError(f"RUN_SEED must be one of {PUBLICATION_SEEDS}, got {RUN_SEED}")

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
MODEL_REVISION = "246e42c643bd9653381dafe9cd5a2b2eb51944b7"

# Frozen old-protocol Qwen Gated validation-selected checkpoints.
GATED_BEST_STEP_BY_SEED = {
    42: 800,
    1337: 932,
    2026: 800,
    31415: 932,
    271828: 932,
}

SHARED_FORWARD_PROTOCOL_SHA256 = "37b7d30a994fb98225d49d76cca5768a838d456abf844dc1a5d93019a0cd15aa"
PARENT_GATED_PROTOCOL_SHA256 = "627bdf5a77ab694581ff590c32f8885c901ebad09e8df86dbab07762caaf95dd"
FACETGATE_PROTOCOL_SHA256 = "735b3e08413949cdb148fe4ce570493b5cd410e562dad7a12748b2e5f2fd6a38"

print({
    "RUN_SEED": RUN_SEED,
    "expected_old_gated_best_step": GATED_BEST_STEP_BY_SEED[RUN_SEED],
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "locked_test": False,
})


In [ ]:
# CELL 2 — CPU-ONLY SAME-SEED OLD-GATED DISCOVERY
# No torch/CUDA import is needed here.
import json, os, re, hashlib, zipfile
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
EXPECTED_STEP = int(GATED_BEST_STEP_BY_SEED[RUN_SEED])

def _walk(obj):
    if isinstance(obj, dict):
        yield obj
        for v in obj.values():
            yield from _walk(v)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            yield from _walk(v)

def _seed_values(obj):
    vals = []
    for d in _walk(obj):
        for k, v in d.items():
            if str(k).strip().casefold() in {"seed", "run_seed", "training_seed", "model_seed"}:
                if isinstance(v, (int, float)) and float(v).is_integer():
                    vals.append(int(v))
    return sorted(set(vals))

def _valid_manifest(obj, step):
    if not isinstance(obj, dict):
        return False, "manifest_not_dict"
    if int(obj.get("global_step", -1)) != int(step):
        return False, f"global_step={obj.get('global_step')}"
    mid = obj.get("model_id")
    if mid is not None and str(mid) != MODEL_ID:
        return False, f"model_id={mid}"
    rev = obj.get("model_revision")
    if rev is not None and str(rev) != MODEL_REVISION:
        return False, f"model_revision={rev}"
    seeds = _seed_values(obj)
    if seeds != [int(RUN_SEED)]:
        return False, f"seed_provenance={seeds}"
    # If the parent protocol is explicitly recorded, it must match.
    text = json.dumps(obj, sort_keys=True, default=str)
    if "gated" not in text.casefold():
        return False, "no_gated_provenance"
    return True, "ok"

def _checkpoint_files_ok(cp):
    cp = Path(cp)
    return (
        (cp / "classifier_heads.pt").is_file()
        and (cp / "adapter" / "adapter_config.json").is_file()
        and (
            (cp / "adapter" / "adapter_model.safetensors").is_file()
            or (cp / "adapter" / "adapter_model.bin").is_file()
        )
        and (cp / "checkpoint_manifest.json").is_file()
    )

def _threshold_near_checkpoint(cp):
    cp = Path(cp)
    roots = []
    if cp.parent.name == "checkpoints":
        roots.append(cp.parent.parent)
    roots.extend([cp.parent, cp.parent.parent])
    seen = set()
    for root in roots:
        try:
            root = root.resolve()
        except Exception:
            pass
        if str(root) in seen or not root.exists():
            continue
        seen.add(str(root))
        for rel in [
            Path("artifacts/thresholds.json"),
            Path("thresholds.json"),
            Path("artifacts/validation_thresholds.json"),
            Path("validation_thresholds.json"),
        ]:
            p = root / rel
            if p.is_file():
                try:
                    obj = json.loads(p.read_text(encoding="utf-8"))
                    if isinstance(obj, dict):
                        return p
                except Exception:
                    pass
    return None

def _direct_candidates():
    rows = []
    if not INPUT_ROOT.exists():
        return rows
    names = {f"checkpoint-{EXPECTED_STEP}", f"checkpoint-{EXPECTED_STEP:04d}"}
    for mf in INPUT_ROOT.rglob("checkpoint_manifest.json"):
        cp = mf.parent
        if cp.name not in names:
            continue
        if not _checkpoint_files_ok(cp):
            continue
        try:
            obj = json.loads(mf.read_text(encoding="utf-8"))
        except Exception as e:
            rows.append({"status":"rejected","source":str(cp),"reason":f"manifest_unreadable:{type(e).__name__}"})
            continue
        ok, reason = _valid_manifest(obj, EXPECTED_STEP)
        if not ok:
            rows.append({"status":"rejected","source":str(cp),"reason":reason})
            continue
        th = _threshold_near_checkpoint(cp)
        if th is None:
            rows.append({"status":"rejected","source":str(cp),"reason":"same-output frozen thresholds not found"})
            continue
        fingerprint = obj.get("checkpoint_content_sha256")
        if not fingerprint:
            fingerprint = hashlib.sha256(
                ((cp/"classifier_heads.pt").read_bytes()[:1<<20])
                + str(cp).encode()
            ).hexdigest()
        rows.append({
            "status":"valid","kind":"directory","source":str(cp),
            "thresholds":str(th),"fingerprint":str(fingerprint),
            "manifest":str(mf),
        })
    return rows

def _zip_candidates():
    rows = []
    if not INPUT_ROOT.exists():
        return rows
    cp_names = [f"checkpoint-{EXPECTED_STEP}/", f"checkpoint-{EXPECTED_STEP:04d}/"]
    for zp in INPUT_ROOT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(zp) as zf:
                names = zf.namelist()
                manifest_members = [
                    n for n in names
                    if n.endswith("checkpoint_manifest.json")
                    and any(tag in n for tag in cp_names)
                ]
                for mm in manifest_members:
                    prefix = mm[:-len("checkpoint_manifest.json")]
                    required = [
                        prefix+"classifier_heads.pt",
                        prefix+"adapter/adapter_config.json",
                    ]
                    adapter_ok = (
                        prefix+"adapter/adapter_model.safetensors" in names
                        or prefix+"adapter/adapter_model.bin" in names
                    )
                    if not all(x in names for x in required) or not adapter_ok:
                        continue
                    obj = json.loads(zf.read(mm).decode("utf-8"))
                    ok, reason = _valid_manifest(obj, EXPECTED_STEP)
                    if not ok:
                        rows.append({"status":"rejected","source":str(zp),"reason":reason})
                        continue
                    threshold_members = [
                        n for n in names
                        if n.endswith("/artifacts/thresholds.json")
                        or n.endswith("/thresholds.json")
                        or n.endswith("/artifacts/validation_thresholds.json")
                        or n.endswith("/validation_thresholds.json")
                    ]
                    if not threshold_members:
                        rows.append({"status":"rejected","source":str(zp),"reason":"thresholds_not_in_zip"})
                        continue
                    # Extract only the threshold JSON; the target child can extract the checkpoint itself.
                    member = sorted(threshold_members, key=len)[0]
                    out = WORK_ROOT / f"_qwen_old_gated_seed{RUN_SEED}_thresholds.json"
                    out.write_bytes(zf.read(member))
                    fingerprint = obj.get("checkpoint_content_sha256") or hashlib.sha256((str(zp)+prefix).encode()).hexdigest()
                    rows.append({
                        "status":"valid","kind":"zip","source":str(zp),
                        "thresholds":str(out),"fingerprint":str(fingerprint),
                        "manifest_member":mm,
                    })
        except Exception:
            continue
    return rows

discovery_rows = _direct_candidates() + _zip_candidates()
valid = [r for r in discovery_rows if r.get("status") == "valid"]

# Collapse byte-identical/manifest-identical duplicate copies.
unique = {}
for r in valid:
    unique.setdefault(r["fingerprint"], r)
valid_unique = list(unique.values())

if len(valid_unique) > 1:
    raise RuntimeError(
        "More than one different valid same-seed old-Gated parent was found. "
        "Remove the ambiguity before running.\n" + json.dumps(valid_unique, indent=2)
    )

PARENT = valid_unique[0] if valid_unique else None
EXECUTION_MODE = "REUSE_EXISTING_GATED" if PARENT else "SIMULTANEOUS_FROM_SCRATCH"

controller_record = {
    "schema":"M3Offend_Qwen_SharedForward_Controller_v1",
    "seed":int(RUN_SEED),
    "model_id":MODEL_ID,
    "model_revision":MODEL_REVISION,
    "expected_old_gated_best_step":EXPECTED_STEP,
    "execution_mode":EXECUTION_MODE,
    "parent":PARENT,
    "valid_parent_count":len(valid_unique),
    "rejected_candidates":[r for r in discovery_rows if r.get("status")=="rejected"][:100],
    "locked_test":False,
    "shared_forward_protocol_sha256":SHARED_FORWARD_PROTOCOL_SHA256,
}
(WORK_ROOT / f"qwen_shared_forward_controller_seed{RUN_SEED}.json").write_text(
    json.dumps(controller_record, indent=2), encoding="utf-8"
)
print("[CONTROLLER DECISION]", json.dumps(controller_record, indent=2))


In [ ]:
# CELL 3 — EMBEDDED REVIEWED CHILD NOTEBOOKS (compressed)
# These are source templates, not external dependencies.
SIM_CHILD_ZLIB_B64 = '...'
TARGET_ONLY_CHILD_ZLIB_B64 = '...'

SIM_TEMPLATE_SHA256 = 'c29862df87853286f01dac9f95005b56a71d116a47ce4839d94de08b994583df'
TARGET_TEMPLATE_SHA256 = 'b36be358df880c101993011ce0820bd03e304158d4e395d0054325ad545c74cf'

print({
    "simultaneous_template_sha256": SIM_TEMPLATE_SHA256,
    "target_only_template_sha256": TARGET_TEMPLATE_SHA256,
})


In [ ]:
# CELL 4 — MATERIALIZE THE SELECTED CHILD WITH EXACT SEED/PARENT BINDING
import base64, zlib, json, hashlib
from pathlib import Path

def _decode_template(blob, expected_sha):
    raw = zlib.decompress(base64.b64decode(blob.encode("ascii")))
    observed = hashlib.sha256(raw).hexdigest()
    if observed != expected_sha:
        raise RuntimeError(f"Embedded child template SHA mismatch: expected={expected_sha} observed={observed}")
    return json.loads(raw.decode("utf-8"))

if EXECUTION_MODE == "REUSE_EXISTING_GATED":
    child = _decode_template(TARGET_ONLY_CHILD_ZLIB_B64, TARGET_TEMPLATE_SHA256)
else:
    child = _decode_template(SIM_CHILD_ZLIB_B64, SIM_TEMPLATE_SHA256)

for cell in child.get("cells", []):
    if cell.get("cell_type") != "code":
        continue
    src = "".join(cell.get("source", []))
    src = src.replace("__RUN_SEED__", str(int(RUN_SEED)))
    if EXECUTION_MODE == "REUSE_EXISTING_GATED":
        src = src.replace(
            'GATED_SOURCE_OVERRIDE=""',
            f'GATED_SOURCE_OVERRIDE={PARENT["source"]!r}'
        )
        src = src.replace(
            'GATED_THRESHOLDS_OVERRIDE=""',
            f'GATED_THRESHOLDS_OVERRIDE={PARENT["thresholds"]!r}'
        )
    cell["source"] = src.splitlines(keepends=True)
    cell["execution_count"] = None
    cell["outputs"] = []

child_name = (
    f"qwen_seed{RUN_SEED}_frozen_parent_target_only.ipynb"
    if EXECUTION_MODE == "REUSE_EXISTING_GATED"
    else f"qwen_seed{RUN_SEED}_shared_forward_simultaneous.ipynb"
)
CHILD_NOTEBOOK = WORK_ROOT / child_name
CHILD_NOTEBOOK.write_text(json.dumps(child, ensure_ascii=False, indent=1), encoding="utf-8")

# Compile every code cell before execution.
for i, cell in enumerate(child["cells"]):
    if cell.get("cell_type") == "code":
        compile("".join(cell.get("source", [])), f"{child_name}:cell{i}", "exec")

CHILD_SHA256 = hashlib.sha256(CHILD_NOTEBOOK.read_bytes()).hexdigest()
print("[CHILD STATIC QA PASS]", {
    "mode":EXECUTION_MODE,
    "path":str(CHILD_NOTEBOOK),
    "sha256":CHILD_SHA256,
    "seed":RUN_SEED,
})


In [ ]:
# CELL 5 — EXECUTE THE SELECTED PIPELINE IN A FRESH CHILD KERNEL
# The controller itself remains CPU-only. The child owns the GPU/torchrun lifecycle.
import subprocess, sys, os, time, json
from pathlib import Path

EXECUTED_NOTEBOOK = CHILD_NOTEBOOK.with_name(CHILD_NOTEBOOK.stem + "_EXECUTED.ipynb")
cmd = [
    sys.executable, "-m", "jupyter", "nbconvert",
    "--to", "notebook",
    "--execute",
    "--ExecutePreprocessor.timeout=-1",
    "--output", EXECUTED_NOTEBOOK.name,
    str(CHILD_NOTEBOOK),
]
print("[EXECUTING]", " ".join(cmd), flush=True)
started = time.time()
proc = subprocess.run(cmd, cwd=str(WORK_ROOT))
elapsed = time.time() - started
if proc.returncode != 0:
    raise RuntimeError(
        f"Selected child pipeline failed (mode={EXECUTION_MODE}, returncode={proc.returncode}). "
        f"Inspect {EXECUTED_NOTEBOOK if EXECUTED_NOTEBOOK.exists() else CHILD_NOTEBOOK}."
    )

print("[PIPELINE EXECUTION COMPLETE]", {
    "mode":EXECUTION_MODE,
    "seed":RUN_SEED,
    "elapsed_seconds":elapsed,
    "executed_notebook":str(EXECUTED_NOTEBOOK),
})


In [ ]:
# CELL 6 — FINAL MODE/ARTIFACT SUMMARY
import json
from pathlib import Path

if EXECUTION_MODE == "REUSE_EXISTING_GATED":
    expected_root = Path(f"/kaggle/working/M3Offend_9318_FacetGate_Qwen3VL8B_Seed{RUN_SEED}")
    result_candidates = list(expected_root.glob("*RESULTS*.zip")) if expected_root.exists() else []
    print("[FINAL]", {
        "mode":"REUSE_EXISTING_GATED",
        "meaning":"old same-seed Gated checkpoint reused/frozen; only SHTE/FacetGate Target trained",
        "result_root":str(expected_root),
        "result_zips":[str(x) for x in result_candidates],
        "locked_test_forwards":0,
    })
else:
    expected_root = Path(f"/kaggle/working/m3offend_9318_qwen3_shared_forward_dual_checkpoint_seed{RUN_SEED}")
    manifest = expected_root / "artifacts" / "dual_checkpoint_final_manifest.json"
    payload = json.loads(manifest.read_text(encoding="utf-8")) if manifest.exists() else None
    print("[FINAL]", {
        "mode":"SIMULTANEOUS_FROM_SCRATCH",
        "meaning":"one Qwen forward per training micro-batch; Gated and detached SHTE updated in the same outer loop",
        "result_root":str(expected_root),
        "dual_checkpoint_manifest":str(manifest) if manifest.exists() else None,
        "selection":payload,
        "locked_test_forwards":0,
    })
